In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import cellrank as cr

In [ ]:
# =========================
# INPUT
# =========================
H5AD = "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/05b_trajectory_revisit/adata_CD8_subset_clusters_0_1_2_4.h5ad"

# =========================
# OUTPUT DIR (DE_DIR)
# =========================
# 你后续 driver gene 脚本用的是 DE_DIR 里这些 CSV
DE_DIR = "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/10_TF_analysis"
os.makedirs(DE_DIR, exist_ok=True)

CLUSTER_COL = 'leiden_T_0.6_relabel'   # e.g. "leiden" or "cluster" or "leiden_cd8"
RESPONSE_COL = 'Respond'  # e.g. "Responder" or "response" or "RNR" or "is_responder"

print("DE_DIR:", DE_DIR)




In [ ]:
'''
import shutil, os

H5AD_BAK = H5AD + ".bak"

if not os.path.exists(H5AD_BAK):
    shutil.copy2(H5AD, H5AD_BAK)
    print("Backup saved:", H5AD_BAK)
else:
    print("Backup exists:", H5AD_BAK)
'''

In [ ]:
'''
import h5py


with h5py.File(H5AD, "r+") as f:
    grp_path = "uns/rank_genes_groups/params"
    if grp_path in f:
        grp = f[grp_path]
        if "layer" in grp:
            del grp["layer"]
            print("Deleted:", grp_path + "/layer")
        else:
            print("No 'layer' key found under", grp_path)
    else:
        print("No group found:", grp_path)
'''


In [ ]:
import scanpy as sc
ad = sc.read_h5ad(H5AD)
ad


In [ ]:
old_key = "leiden_T_0.6"
new_key = "leiden_T_0.6_relabel"

# 映射：old -> new
mapping = {
    "4": "1",
    "0": "2",
    "1": "3",
    "2": "4",
    }

# 确保是字符串（scanpy 的 leiden 通常本来就是字符串）
ad.obs[old_key] = ad.obs[old_key].astype(str)

# 生成新标签；未在 mapping 里的（比如 7/8）会变成 NaN
ad.obs[new_key] = ad.obs[old_key].map(mapping)

# 可选：把没映射到的保留原值（例如 7/8），避免 NaN
ad.obs[new_key] = ad.obs[new_key].fillna(ad.obs[old_key])

In [ ]:
print(ad.obs['leiden_T_0.6_relabel'])
print(ad.obs_names)

In [ ]:
df = pd.DataFrame({
    "cell_name": ad.obs_names.astype(str),
    "leiden_T_0.6_relabel": ad.obs["leiden_T_0.6_relabel"].astype(str).values
})

out_csv = "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/02_extracting_T_cells_and_clustering/cell_leiden_T_0.6_relabel.csv"
df.to_csv(out_csv, index=False)

In [ ]:
fixed_path = H5AD.replace(".h5ad", ".fixed.h5ad")
ad.write_h5ad(fixed_path, compression="gzip")
print("Saved fixed file:", fixed_path)

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp

def make_expressed_tf_list(
    adata,
    tf_all_path: str,
    out_path: str,
    min_pct: float = 0.02,   # 2% cells
    min_cells: int = 30,     # at least 30 cells
    layer: str | None = None # e.g. "counts"; if None uses adata.X
):
    tfs = pd.read_csv(tf_all_path, header=None)[0].astype(str).str.upper().unique()
    genes = pd.Index(adata.var_names.astype(str).str.upper())

    X = adata.layers[layer] if (layer is not None and layer in adata.layers) else adata.X
    if sp.issparse(X):
        nnz = np.array((X > 0).sum(axis=0)).ravel()
    else:
        nnz = (X > 0).sum(axis=0)

    thr = max(min_cells, int(np.ceil(min_pct * adata.n_obs)))
    expressed_genes = set(genes[nnz >= thr])

    keep = sorted(set(tfs).intersection(expressed_genes))
    pd.Series(keep).to_csv(out_path, index=False, header=False)
    print(f"[OK] TFs kept: {len(keep)} (threshold: >= {thr} cells). Saved to {out_path}")

# 用法：
make_expressed_tf_list(ad, "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/10_TF_analysis/pyscenic_assets/tfs/hg38_tfs_all.txt", "tfs_cd8_expressed.txt", min_pct=0.02, layer="counts")


In [ ]:
ad

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

def filter_tfs_by_hvg(adata, tf_list_path, out_path, n_top=5000, flavor="seurat_v3", layer="counts"):
    tfs = pd.read_csv(tf_list_path, header=None)[0].astype(str).str.upper().unique()
    genes = adata.var_names.astype(str)
    gene_upper = pd.Index([g.upper() for g in genes])

    # HVG on a copy to avoid overwriting var
    ad = adata.copy()
    if layer in ad.layers:
        sc.pp.highly_variable_genes(ad, n_top_genes=n_top, flavor=flavor, layer=layer, subset=False)
    else:
        sc.pp.highly_variable_genes(ad, n_top_genes=n_top, flavor=flavor, subset=False)

    hvg = set(gene_upper[ad.var["highly_variable"].values])
    keep = sorted(set(tfs).intersection(hvg))

    pd.Series(keep).to_csv(out_path, index=False, header=False)
    print(f"[OK] kept TFs: {len(keep)} (HVG-based). saved: {out_path}")

# 用法：
filter_tfs_by_hvg(ad, "tfs_cd8_expressed.txt", "tfs_cd8_hvg600.txt", n_top=5000)
# 说明：n_top 是在全基因里选 HVG 的数量，最后再和 TF 取交集，TF 会落到 ~300-800


In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

import os
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

def export_pyscenic_expr_from_counts(
    adata,
    out_expr_tsv_gz: str,
    tf_list_path: str | None = None,   # 传入你的 tfs_cd8_hvg600.txt（可选但推荐）
    n_hvg: int = 5000,                 # 只保留 5000 HVG
    target_sum: float = 1e4,
    counts_layer: str = "counts",
    hvg_flavor: str = "seurat_v3",
    make_gene_upper: bool = True,
):
    """
    Export pySCENIC expression matrix (cells x genes, TSV.GZ) from counts.

    Pipeline:
      1) compute HVG on counts (n_hvg)
      2) keep genes = HVG ∪ TFs (if tf_list_path provided)
      3) normalize_total(target_sum) + log1p on counts
      4) export

    Notes:
      - Keeps TFs even if not in HVG.
      - Uppercases gene symbols in the output to match AertsLab TF/motif resources.
    """
    os.makedirs(os.path.dirname(out_expr_tsv_gz), exist_ok=True)

    # ---------- 1) HVG on counts ----------
    tmp = adata.copy()
    Xc = tmp.layers[counts_layer] if counts_layer in tmp.layers else tmp.X
    tmp.X = Xc.tocsr() if sp.issparse(Xc) else np.asarray(Xc)

    sc.pp.highly_variable_genes(
        tmp,
        n_top_genes=n_hvg,
        flavor=hvg_flavor,
        layer=None,   # because counts already in X
        subset=False,
    )

    # Gene name normalization for matching
    var_upper = pd.Index(adata.var_names.astype(str)).str.upper()
    hvg_upper = set(pd.Index(tmp.var_names[tmp.var["highly_variable"].values].astype(str)).str.upper())

    # ---------- 2) optional TF union ----------
    tf_upper = set()
    if tf_list_path is not None:
        tfs = pd.read_csv(tf_list_path, header=None)[0].astype(str)
        tf_upper = set(pd.Index(tfs).str.upper())

    keep_upper = hvg_upper.union(tf_upper)
    keep_mask = var_upper.isin(keep_upper)

    ad = adata[:, keep_mask].copy()
    print(f"[INFO] genes kept: {ad.n_vars} (HVG={len(hvg_upper)} + TF={len(tf_upper)})")

    # ---------- 3) normalize + log1p from counts ----------
    X = ad.layers[counts_layer]
    ad.X = X.tocsr() if sp.issparse(X) else np.asarray(X)

    sc.pp.normalize_total(ad, target_sum=target_sum)
    sc.pp.log1p(ad)

    # ---------- 4) export ----------
    genes = ad.var_names.astype(str)
    if make_gene_upper:
        genes = pd.Index(genes).str.upper()

    if sp.issparse(ad.X):
        df = pd.DataFrame.sparse.from_spmatrix(ad.X.tocsr(), index=ad.obs_names.astype(str), columns=genes)
    else:
        df = pd.DataFrame(np.asarray(ad.X), index=ad.obs_names.astype(str), columns=genes)

    df.to_csv(out_expr_tsv_gz, sep="\t", compression="gzip")
    print(f"[OK] saved {out_expr_tsv_gz} shape={df.shape}")
    return out_expr_tsv_gz

expr_out = "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/10_TF_analysis/pyscenic_run/cd8/expr_log1pCPM.tsv.gz"
tf_path  = "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/10_TF_analysis/tfs_cd8_hvg600.txt"

export_pyscenic_expr_from_counts(
    ad,            # 你的 CD8 adata 变量
    expr_out,
    tf_list_path=tf_path, # 强制保留 TF
    n_hvg=5000,
    counts_layer="counts",
)

In [ ]:
import pandas as pd
import scanpy as sc

AUC = "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/10_TF_analysis/pyscenic_run/cd8/auc.tsv.gz"

auc = pd.read_csv(AUC, sep="\t", index_col=0)
auc = auc.reindex(ad.obs_names)

ad.obsm["X_regulon_auc"] = auc.values
ad.uns["regulon_auc_cols"] = auc.columns.tolist()

out = H5AD.replace(".h5ad", ".scenic_fast.h5ad")
ad.write_h5ad(out, compression="gzip")
print("saved:", out, "AUC:", auc.shape)


In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.stats import ranksums
from statsmodels.stats.multitest import multipletests

def diff_regulon_auc(auc_df: pd.DataFrame, group: pd.Series, g1, g2, min_cells=30):
    """
    auc_df: cells x regulons
    group:  cell -> group label
    g1/g2:  two group labels
    """
    idx1 = group.index[(group == g1).values]
    idx2 = group.index[(group == g2).values]

    if len(idx1) < min_cells or len(idx2) < min_cells:
        raise ValueError(f"Too few cells: {g1}={len(idx1)}, {g2}={len(idx2)} (min={min_cells})")

    X1 = auc_df.loc[idx1]
    X2 = auc_df.loc[idx2]

    # effect size: mean diff + log2FC (mean ratio, with small eps)
    eps = 1e-9
    mean1 = X1.mean(axis=0)
    mean2 = X2.mean(axis=0)
    delta = mean2 - mean1
    log2fc = np.log2((mean2 + eps) / (mean1 + eps))

    # stats
    pvals = []
    for col in auc_df.columns:
        pvals.append(ranksums(X1[col].values, X2[col].values).pvalue)
    pvals = np.asarray(pvals)

    qvals = multipletests(pvals, method="fdr_bh")[1]

    res = pd.DataFrame({
        "regulon": auc_df.columns,
        "n_g1": len(idx1),
        "n_g2": len(idx2),
        "mean_g1": mean1.values,
        "mean_g2": mean2.values,
        "delta_g2_minus_g1": delta.values,
        "log2fc_g2_over_g1": log2fc.values,
        "pval": pvals,
        "qval": qvals,
    }).sort_values(["qval", "pval"])

    return res


In [ ]:
STATE_COL = "leiden_T_0.6_relabel"      # 改：你 C1..C7 的列名（例如 "leiden_T_0.6_relabel" 或你自己定义的）
RESPONSE_COL = "Respond"    # 改：R/NR 列名（比如 True/False, "R"/"NR", "Responder"/"Non-responder"）


In [ ]:
mask_C3

In [ ]:
# C2 vs C3
res_C2_C3 = diff_regulon_auc(auc, ad.obs[STATE_COL].astype(str), "2", "3", min_cells=30)
res_C2_C3.to_csv("regulon_diff_C2_vs_C3.csv", index=False)

# C4 vs C3
res_C4_C3 = diff_regulon_auc(auc, ad.obs[STATE_COL].astype(str), "4", "3", min_cells=30)
res_C4_C3.to_csv("regulon_diff_C4_vs_C3.csv", index=False)

# C3: NR vs R（先子集到 C3）
mask_C3 = ad.obs[STATE_COL].astype(str) == "3"
auc_C3 = auc.loc[ad.obs_names[mask_C3]]
resp_C3 = ad.obs.loc[mask_C3, RESPONSE_COL].astype(str)

# 你响应标签如果是 0/1 或 True/False，这里统一成 "NR"/"R"
# 你自己按实际情况映射一下：
# resp_C3 = resp_C3.map({0:"NR", 1:"R"})
# resp_C3 = resp_C3.map({"Non-responder":"NR","Responder":"R"})

res_C3_NR_R = diff_regulon_auc(auc_C3, resp_C3, "Non-Responder", "Responder", min_cells=15)
res_C3_NR_R.to_csv("regulon_diff_C3_NR_vs_R.csv", index=False)

display(res_C2_C3)
display(res_C4_C3.head(10))
display(res_C3_NR_R.head(10))


In [ ]:
import matplotlib.pyplot as plt

def plot_top_heatmap(auc_df, group, g1, g2, res_df, top_n=30, title=None):
    top = res_df.sort_values("qval").head(top_n)["regulon"].tolist()

    idx = group.index[(group == g1) | (group == g2)]
    sub = auc_df.loc[idx, top].copy()
    sub["__group__"] = group.loc[idx].values

    # 排序：g1 在前，g2 在后；组内按第一个 regulon 排序让图更稳定
    sub = sub.sort_values(["__group__", top[0]], ascending=[True, False])

    mat = sub[top].to_numpy()
    # 行标准化（z-score），更适合热图展示
    mu = mat.mean(axis=0, keepdims=True)
    sd = mat.std(axis=0, keepdims=True) + 1e-9
    z = (mat - mu) / sd

    plt.figure(figsize=(0.25*top_n + 4, 10))
    plt.imshow(z, aspect="auto", interpolation="nearest")
    plt.yticks([])
    plt.xticks(range(top_n), top, rotation=90)
    plt.colorbar()
    plt.title(title or f"Top {top_n} regulons: {g2} vs {g1}")
    plt.tight_layout()
    plt.show()

# C2 vs C3
plot_top_heatmap(auc, ad.obs[STATE_COL].astype(str), "2", "3", res_C2_C3, top_n=30, title="C3 vs C2 (AUC)")

# C4 vs C3
plot_top_heatmap(auc, ad.obs[STATE_COL].astype(str), "4", "3", res_C4_C3, top_n=30, title="C3 vs C4 (AUC)")

# C3: NR vs R
plot_top_heatmap(auc_C3, resp_C3, "Non-Responder", "Responder", res_C3_NR_R, top_n=30, title="C3: R vs NR (AUC)")


In [ ]:
ad

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import scanpy as sc
import cellrank as cr
import matplotlib.pyplot as plt
import matplotlib as mpl
CLUSTER_KEY = "leiden_T_0.6_relabel"
ROOT_CLUSTER = "1"
ROOT_STRATEGY = "top_naive"
NAIVE_KEY = "score_Naive"   # 你已有的 naive 分数列名；如果没有就用下面方法计算
NAIVE_TOP_FRAC = 0.10       # 前10%
NAIVE_MIN_CAND = 30         # 候选太少时的保底（按你数据量可改）
# GPCCA parameters
N_COMPONENTS = 20
N_STATES_RANGE = (4, 10)        # lets minChi pick
N_TERMINAL = 3                  # top_n terminal macrostates

# Branch hard-label parameters (relative advantage)
DELTA = 0.15  # fp_A - fp_B >= DELTA
EPS   = 0.25  # fp_A (or fp_B) >= EPS

In [ ]:
def pick_root_cell(ad: sc.AnnData, rep: str) -> int:
    """Pick a robust root cell (iroot) using a cluster-defined candidate set."""
    mask_root = (ad.obs[CLUSTER_KEY].astype(str) == str(ROOT_CLUSTER)).values
    idxs = np.where(mask_root)[0]
    if len(idxs) == 0:
        raise ValueError(f"No cells found in ROOT_CLUSTER={ROOT_CLUSTER} for {CLUSTER_KEY}.")

    # ----- candidate selection: top naive within root cluster -----
    if ROOT_STRATEGY == "top_naive":
        if NAIVE_KEY not in ad.obs.columns:
            raise ValueError(f"ROOT_STRATEGY=top_naive but {NAIVE_KEY} not in ad.obs.")
        sub = ad.obs.loc[mask_root, NAIVE_KEY].astype(float)

        # take top 10% (or at least NAIVE_MIN_CAND)
        n_cand = max(int(np.ceil(len(sub) * NAIVE_TOP_FRAC)), NAIVE_MIN_CAND)
        n_cand = min(n_cand, len(sub))
        cand_names = sub.nlargest(n_cand).index
        cand_idxs = np.where(ad.obs_names.isin(cand_names))[0]

        print(f"[INFO] Root candidates: top {n_cand}/{len(sub)} ({NAIVE_TOP_FRAC:.0%}) by {NAIVE_KEY} in cluster {ROOT_CLUSTER}.")
    else:
        # fallback: all cells in root cluster
        cand_idxs = idxs
        print(f"[INFO] Root candidates: all {len(cand_idxs)} cells in cluster {ROOT_CLUSTER}.")

    # ----- pick representation space for medoid -----
    X = ad.obsm[rep][cand_idxs]

    # ----- medoid: most central cell among candidates -----
    x2 = np.sum(X * X, axis=1, keepdims=True)
    D2 = x2 + x2.T - 2 * (X @ X.T)
    D2 = np.maximum(D2, 0.0)
    medoid_local = int(np.argmin(D2.mean(axis=1)))
    iroot = int(cand_idxs[medoid_local])

    print(f"[INFO] Root cell selected by medoid in {rep}: {ad.obs_names[iroot]} (iroot={iroot})")
    return iroot

def ensure_neighbors_for_dpt(ad: sc.AnnData):
    """
    DPT needs connectivities. If neighbors missing, compute from X_scVI if present,
    otherwise from PCA.
    """
    if "neighbors" in ad.uns and "connectivities" in ad.obsp:
        return

    if "X_scVI" in ad.obsm:
        print("[INFO] Computing neighbors using X_scVI.")
        sc.pp.neighbors(ad, use_rep="X_scVI", n_neighbors=50)
    else:
        print("[INFO] Computing PCA + neighbors (X_pca).")
        if "X_pca" not in ad.obsm:
            sc.pp.pca(ad, n_comps=50)
        sc.pp.neighbors(ad, use_rep="X_pca", n_neighbors=50)

def infer_lineage_for_cluster(fp_df: pd.DataFrame, ad: sc.AnnData, cluster_value: str) -> str:
    """
    Find which lineage (column in fp_df) is most associated with a given cluster,
    by mean fate probability within that cluster.
    """
    mask = (ad.obs[CLUSTER_KEY].astype(str) == str(cluster_value)).values
    means = fp_df.loc[ad.obs_names[mask]].mean(axis=0)
    return str(means.idxmax())

In [ ]:
from scipy import sparse
NAIVE_GENES = [
    "TCF7", "LEF1", "CCR7", "IL7R", "LTB", "MAL", "SELL",
    "TRAC", "CD3D", "CD3E"
]

genes = [g for g in NAIVE_GENES if g in ad.var_names]

# 1) 构建 log1p(scvi_norm_expr) layer（安全处理 sparse）
X = ad.layers["norm_expr"]
if sparse.issparse(X):
    X_log = X.copy()
    X_log.data = np.log1p(X_log.data)
else:
    X_log = np.log1p(X)

ad.layers["norm_expr_log1p"] = X_log

# 2) 用这个 layer 算 score
Xbak = ad.X
ad.X = ad.layers["norm_expr_log1p"]
sc.tl.score_genes(ad, gene_list=genes, score_name=NAIVE_KEY, use_raw=False)
ad.X = Xbak

In [ ]:
sc.pp.neighbors(ad, use_rep="X_scVI", n_neighbors=50)
#ensure_neighbors_for_dpt(ad)
iroot = pick_root_cell(ad, rep="X_scVI")
ad.uns["iroot"] = iroot

# Diffmap + DPT
sc.tl.diffmap(ad)
sc.tl.dpt(ad)  # uses ad.uns["iroot"]
if "dpt_pseudotime" not in ad.obs:
    raise RuntimeError("scanpy did not create ad.obs['dpt_pseudotime'].")

sc.pl.umap(ad, color="dpt_pseudotime", show=True)


In [ ]:
def pick_top_frac_in_cluster(ad, cluster_value, frac=0.05, min_n=50):
    mask = (ad.obs[CLUSTER_KEY].astype(str) == str(cluster_value)).values
    idx = np.where(mask)[0]
    if idx.size == 0:
        raise ValueError(f"No cells in cluster {cluster_value}")
    t = ad.obs['dpt_pseudotime'].values[idx].astype(float)
    thr = np.quantile(t, 1.0 - frac)
    pick = idx[t >= thr]
    if pick.size < min_n:
        # 兜底：取 top min_n（避免太小导致不稳定）
        pick = idx[np.argsort(t)[-min_n:]]
    return np.array(pick, dtype=int)

A_idx = pick_top_frac_in_cluster(ad, cluster_value="4", frac=0.05, min_n=50)  # terminal A from cluster 0
B_idx = pick_top_frac_in_cluster(ad, cluster_value="3", frac=0.05, min_n=50) 

In [ ]:
ad.obs["is_terminal_A_c0top5"] = 0
ad.obs["is_terminal_B_c1top5"] = 0
ad.obs.iloc[A_idx, ad.obs.columns.get_loc("is_terminal_A_c0top5")] = 1
ad.obs.iloc[B_idx, ad.obs.columns.get_loc("is_terminal_B_c1top5")] = 1

In [ ]:
pk = cr.kernels.PseudotimeKernel(ad, time_key='dpt_pseudotime')
pk.compute_transition_matrix()
T=pk.transition_matrix.tocsr()

In [ ]:
n = T.shape[0]

A = np.zeros(n, dtype=float); A[A_idx] = 1.0
B = np.zeros(n, dtype=float); B[B_idx] = 1.0

def kstep_mass(T, indicator, k):
    v = indicator.copy()
    for _ in range(k):
        v = T @ v
    return np.asarray(v).ravel()

for k in [1, 5, 10, 20]:
    uA = kstep_mass(T, A, k)
    uB = kstep_mass(T, B, k)
    ad.obs[f"p_in_A_step{k}"] = uA
    ad.obs[f"p_in_B_step{k}"] = uB
    print(f"k={k}: mean(A)={uA.mean():.4g}, mean(B)={uB.mean():.4g}, frac(A>B)={(uA>uB).mean():.3f}")

# 画一下 k=10 的差值在 UMAP 上（看分叉是否存在）

ad.obs["k10_diff_AminusB"] = ad.obs["p_in_A_step10"] - ad.obs["p_in_B_step10"]
sc.pl.umap(ad, color=["k10_diff_AminusB"], show=False)
plt.show()

# 看分布
plt.figure(figsize=(5.2,3.6))
plt.hist(ad.obs["k10_diff_AminusB"], bins=60, alpha=0.8)
plt.title("Distribution of k=10 reachability diff (A-B)")
plt.xlabel("p_in_A_step10 - p_in_B_step10")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

# ----------------------------
# Basic matplotlib config
# ----------------------------
mpl.rcParams["font.family"] = "Liberation Sans"
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)



In [ ]:
import numpy as np

x = ad.obs["k10_diff_AminusB"].values.astype(float)

# tau 控制“多快从 0.5 变到两端”，建议用分位数尺度
tau = np.quantile(np.abs(x), 0.7) + 1e-12
s = 1.0 / (1.0 + np.exp(-x / tau))

ad.obs["branch_preference_A_sigmoid"] = s
print("tau:", tau, "saved: branch_preference_A_sigmoid")

In [ ]:
sc.pl.umap(ad, color=["branch_preference_A_sigmoid"], show=False, save="_branch_preference_A_sigmoid.png")
plt.show()

# 看分布
plt.figure(figsize=(5.2,3.6))
plt.hist(ad.obs["branch_preference_A_sigmoid"], bins=60, alpha=0.8)
plt.title("Distribution of k=10 reachability diff (A-B)")
plt.xlabel("p_in_A_step10 - p_in_B_step10")
plt.tight_layout()
plt.show()

In [ ]:
score_key = "branch_preference_A_sigmoid"
out_key   = "branch_AB_0p5"

ad.obs[out_key] = np.where(ad.obs[score_key].values >= 0.46, "branch_A", "branch_B")
ad.obs[out_key] = pd.Categorical(ad.obs[out_key], categories=["branch_A", "branch_B"])

ad.obs[out_key].value_counts()

In [ ]:
import os, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

mpl.rcParams["font.family"] = "Liberation Sans"   # 若无效，换成 "DejaVu Sans"
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

out_png = "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/10_TF_analysis/figures/Fig_branch_assignment_UMAP_AB.png"
os.makedirs(os.path.dirname(out_png), exist_ok=True)

X = ad.obsm["X_umap"]
x, y = X[:, 0], X[:, 1]
lab = ad.obs[out_key].astype(str).values

mA = (lab == "branch_A")
mB = (lab == "branch_B")

fig, ax = plt.subplots(figsize=(6.2, 5.2))

# 背景（所有点）
ax.scatter(x, y, s=3, c="lightgrey", alpha=0.25, linewidths=0, rasterized=True)

# Branch A / B
ax.scatter(x[mA], y[mA], s=7, c="red",  alpha=0.95, linewidths=0, rasterized=True, label=f"A (n={mA.sum()})")
ax.scatter(x[mB], y[mB], s=7, c="blue", alpha=0.95, linewidths=0, rasterized=True, label=f"B (n={mB.sum()})")

ax.set_title("Hard branch assignment", fontsize=14)
ax.set_xlabel("UMAP1", fontsize=12)
ax.set_ylabel("UMAP2", fontsize=12)
ax.legend(frameon=False, fontsize=10, loc="best")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
fig.savefig(out_png, dpi=350, bbox_inches="tight", facecolor="white")
plt.show()
plt.close(fig)

print("[DONE] saved:", out_png)


In [ ]:
sc.pl.umap(ad, color="Respond", show=False)


In [ ]:
ad

In [ ]:
from scipy.stats import rankdata
BRANCH_KEY = "branch_preference_A_sigmoid"   # 你的A/B/other标签
DPT_KEY    = "dpt_pseudotime"
NAIVE_KEY  = "score_Naive"
CYTO_KEY   = "cyto_score"
EXH_KEY    = "exh_score"

maskA = ad.obs[BRANCH_KEY].values >= 0.5 
maskB = ad.obs[BRANCH_KEY].values < 0.5

# --- raw values ---
branch_A_dpt   = ad.obs.loc[maskA, DPT_KEY].astype(float).values
branch_B_dpt   = ad.obs.loc[maskB, DPT_KEY].astype(float).values

In [ ]:
STAT1_top_dn = ['SREBF2', 'MSMO1', 'DHCR24', 'SQLE', 'ACAT2', 'CYP51A1', 'MVD', 'HMGCS1', 'ERG28', 'FDFT1']


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def _get_expr_matrix(ad, genes, use_raw=True, layer=None):
    # 返回 cells x genes 的 numpy
    if use_raw and ad.raw is not None:
        X = ad.raw[:, genes].X
    else:
        X = ad[:, genes].layers[layer] if layer is not None else ad[:, genes].X
    X = X.toarray() if hasattr(X, "toarray") else np.asarray(X)
    return X

def _zscore_cols(M):
    mu = np.nanmean(M, axis=0, keepdims=True)
    sd = np.nanstd(M, axis=0, keepdims=True) + 1e-12
    return (M - mu) / sd

def heatmap_branchA_branchB_stat1_downstream(
    ad,
    genes,
    maskA,
    maskB,
    branch_A_dpt,
    branch_B_dpt,
    out_png,
    max_cells_per_branch=2500,
    use_raw=True,
    layer=None,
    seed=0,
    title="STAT1 downstream genes (Perturb-seq Stim8hr) along Branch A/B pseudotime"
):
    # 1) gene 存在性过滤
    gene_space = ad.raw.var_names if (use_raw and ad.raw is not None) else ad.var_names
    genes_use = [g for g in genes if g in gene_space]
    if len(genes_use) < 5:
        raise ValueError("Too few STAT1 downstream genes found in adata gene space. Check gene symbols / raw.")
    print(f"[INFO] genes used in heatmap: {len(genes_use)} / {len(genes)}")

    # 2) branch 子集并按 dpt 排序
    maskA = np.asarray(maskA).astype(bool)
    maskB = np.asarray(maskB).astype(bool)

    adA = ad[maskA].copy()
    adB = ad[maskB].copy()

    dptA = np.asarray(branch_A_dpt, float)
    dptB = np.asarray(branch_B_dpt, float)

    # 排序
    ordA = np.argsort(dptA)
    ordB = np.argsort(dptB)
    adA = adA[ordA].copy()
    adB = adB[ordB].copy()
    dptA = dptA[ordA]
    dptB = dptB[ordB]

    # 3) 下采样（保持顺序的均匀采样）
    rng = np.random.default_rng(seed)
    if adA.n_obs > max_cells_per_branch:
        idx = np.linspace(0, adA.n_obs - 1, max_cells_per_branch).astype(int)
        adA = adA[idx].copy()
        dptA = dptA[idx]
    if adB.n_obs > max_cells_per_branch:
        idx = np.linspace(0, adB.n_obs - 1, max_cells_per_branch).astype(int)
        adB = adB[idx].copy()
        dptB = dptB[idx]

    # 4) 取表达矩阵并做 per-gene z-score（便于看动态）
    XA = _get_expr_matrix(adA, genes_use, use_raw=use_raw, layer=layer)
    XB = _get_expr_matrix(adB, genes_use, use_raw=use_raw, layer=layer)
    XA = _zscore_cols(XA)
    XB = _zscore_cols(XB)

    # 5) 拼接成一个大矩阵：上半 A，下半 B
    M = np.vstack([XA, XB]).T  # genes x cells
    nA = XA.shape[0]
    nB = XB.shape[0]

    # 6) 画图
    fig = plt.figure(figsize=(12.0, 0.22 * len(genes_use) + 2.8))
    ax = plt.gca()
    im = ax.imshow(M, aspect="auto", interpolation="nearest", cmap="RdBu_r", vmin=-2, vmax=2)

    # 分割线
    ax.axvline(nA - 0.5, color="black", linewidth=1.2)

    # y 轴基因名
    ax.set_yticks(np.arange(len(genes_use)))
    ax.set_yticklabels(genes_use, fontsize=9)

    # x 轴：只标 A/B
    ax.set_xticks([nA/2, nA + nB/2])
    ax.set_xticklabels([f"Branch A (n={nA})", f"Branch B (n={nB})"], fontsize=11)
    ax.tick_params(axis="x", bottom=False, top=False)

    ax.set_title(title, fontsize=13, pad=12)

    cbar = plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02)
    cbar.set_label("Expression (z-score per gene)", rotation=90)

    plt.tight_layout()

    os.makedirs(os.path.dirname(out_png), exist_ok=True)
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    print("[DONE] saved:", out_png)


In [ ]:
out_png = "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/10_TF_analysis/figures/Fig_STAT1_downstream_heatmap_branchAB.png"
genes_stat1 = STAT1_top_dn + STAT1_top_up
heatmap_branchA_branchB_stat1_downstream(
    ad=ad,
    genes=genes_stat1,
    maskA=maskA,
    maskB=maskB,
    branch_A_dpt=branch_A_dpt,
    branch_B_dpt=branch_B_dpt,
    out_png=out_png,
    max_cells_per_branch=2500,
    use_raw=(ad.raw is not None),
    layer='norm_expr_log1p',   # 如果你有 log1p_norm layer，这里改成 "log1p_norm"
    seed=0,
    title="STAT1 perturbation downstream genes (Perturb-seq Stim8hr) show branch-specific dynamics"
)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

def _get_expr_matrix(ad, genes, use_raw=True, layer=None):
    if use_raw and ad.raw is not None:
        X = ad.raw[:, genes].X
    else:
        X = ad[:, genes].layers[layer] if layer is not None else ad[:, genes].X
    X = X.toarray() if hasattr(X, "toarray") else np.asarray(X)
    return X

def _robust_zscore_cols(M):
    # per-gene robust z-score: (x - median) / (1.4826*MAD)
    med = np.nanmedian(M, axis=0, keepdims=True)
    mad = np.nanmedian(np.abs(M - med), axis=0, keepdims=True) + 1e-12
    return (M - med) / (1.4826 * mad)

def _bin_means(dpt, X, n_bins=60, min_n=15):
    """
    dpt: (n_cells,)
    X:   (n_cells, n_genes)
    return:
      centers (n_bins_eff,), Xm (n_bins_eff, n_genes)
    """
    dpt = np.asarray(dpt, float)
    X = np.asarray(X, float)
    m = np.isfinite(dpt)
    dpt = dpt[m]
    X = X[m]

    # 归一化到 0-1（分支内）
    d0, d1 = np.nanmin(dpt), np.nanmax(dpt)
    if d1 <= d0:
        u = np.zeros_like(dpt)
    else:
        u = (dpt - d0) / (d1 - d0)

    edges = np.linspace(0.0, 1.0, n_bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])

    Xm = []
    C = []
    for i in range(n_bins):
        if i < n_bins - 1:
            sel = (u >= edges[i]) & (u < edges[i+1])
        else:
            sel = (u >= edges[i]) & (u <= edges[i+1])
        if sel.sum() < min_n:
            continue
        Xm.append(np.nanmean(X[sel], axis=0))
        C.append(centers[i])

    return np.array(C), np.vstack(Xm)

def heatmap_branchAB_binned_robust(
    ad,
    genes,
    maskA, maskB,
    branch_A_dpt, branch_B_dpt,
    out_png,
    n_bins=60,
    min_n=15,
    use_raw=True,
    layer=None,
    clip_q=(0.02, 0.98),
    title="STAT1 downstream genes (Perturb-seq Stim8hr) show branch-specific dynamics",
):
    # gene filtering
    gene_space = ad.raw.var_names if (use_raw and ad.raw is not None) else ad.var_names
    genes_use = [g for g in genes if g in gene_space]
    if len(genes_use) < 5:
        raise ValueError("Too few genes found in adata gene space.")

    # split
    maskA = np.asarray(maskA).astype(bool)
    maskB = np.asarray(maskB).astype(bool)
    adA = ad[maskA].copy()
    adB = ad[maskB].copy()

    # expression
    XA = _get_expr_matrix(adA, genes_use, use_raw=use_raw, layer=layer)  # cells x genes
    XB = _get_expr_matrix(adB, genes_use, use_raw=use_raw, layer=layer)

    # bin means along within-branch dpt
    cA, XmA = _bin_means(branch_A_dpt, XA, n_bins=n_bins, min_n=min_n)   # bins x genes
    cB, XmB = _bin_means(branch_B_dpt, XB, n_bins=n_bins, min_n=min_n)

    # robust scaling per gene across both branches (拼起来一起做，保证色阶可比)
    Xm = np.vstack([XmA, XmB])   # (binsA+binsB) x genes
    Zm = _robust_zscore_cols(Xm) # robust z

    # clip by quantile for nicer contrast
    lo, hi = np.nanquantile(Zm, clip_q[0]), np.nanquantile(Zm, clip_q[1])
    lo = float(lo); hi = float(hi)
    Zm = np.clip(Zm, lo, hi)

    # reshape to genes x bins
    nA = XmA.shape[0]
    nB = XmB.shape[0]
    ZA = Zm[:nA]
    ZB = Zm[nA:]
    M = np.vstack([ZA, ZB]).T  # genes x (binsA+binsB)

    # ----- plot layout: top dpt bar + main heatmap -----
    fig = plt.figure(figsize=(12.5, 0.22 * len(genes_use) + 2.2))
    gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[0.07, 1.0], hspace=0.03)
    ax_bar = fig.add_subplot(gs[0, 0])
    ax = fig.add_subplot(gs[1, 0])

    # top dpt bar (0->1 each branch)
    # make a 1 x (nA+nB) array: first part cA, second part cB
    bar = np.concatenate([cA, cB])[None, :]
    ax_bar.imshow(bar, aspect="auto", interpolation="nearest", cmap="viridis", vmin=0, vmax=1)
    ax_bar.axvline(nA - 0.5, color="black", linewidth=1.0)
    ax_bar.set_yticks([])
    ax_bar.set_xticks([])
    ax_bar.set_title(title, fontsize=13, pad=6)
    # add small colorbar for dpt
    cax1 = fig.add_axes([0.92, 0.88, 0.010, 0.07])  # [left,bottom,width,height]
    cb1 = plt.colorbar(mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(0,1), cmap="viridis"),
                       cax=cax1)
    cb1.set_label("within-branch dpt (0–1)", rotation=90)

    # main heatmap
    im = ax.imshow(M, aspect="auto", interpolation="nearest", cmap="RdBu_r", vmin=lo, vmax=hi)
    ax.axvline(nA - 0.5, color="black", linewidth=1.2)

    ax.set_yticks(np.arange(len(genes_use)))
    ax.set_yticklabels(genes_use, fontsize=10)

    ax.set_xticks([nA/2, nA + nB/2])
    ax.set_xticklabels([f"Branch A (binned, {nA} bins)", f"Branch B (binned, {nB} bins)"], fontsize=11)
    ax.tick_params(axis="x", bottom=False, top=False)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # expression colorbar
    cax2 = fig.add_axes([0.92, 0.20, 0.015, 0.60])
    cb2 = plt.colorbar(im, cax=cax2)
    cb2.set_label("Expression (robust z-score, clipped)", rotation=90)

    plt.tight_layout(rect=[0, 0, 0.90, 1])  # leave space for colorbars

    os.makedirs(os.path.dirname(out_png), exist_ok=True)
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    print("[DONE] saved:", out_png)


In [ ]:
maskA = ad.obs["branch_preference_A_sigmoid"].values >= 0.46
maskB = ad.obs["branch_preference_A_sigmoid"].values < 0.46
branch_A_dpt = ad.obs.loc[maskA, "dpt_pseudotime"].astype(float).values
branch_B_dpt = ad.obs.loc[maskB, "dpt_pseudotime"].astype(float).values

out_png = "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/10_TF_analysis/figures/Fig_SREBF2_downstream_branchAB_binned_robust.png"
#genes_stat1 = ['IFITM3', 'STAT1', 'ZNF700', 'GBP4', 'GBP1', 'GBP5', 'EPSTI1', 'STX17', 'PARP9', 'IFI35', 'IRF1', 'HAPLN3', 'PSMB9', 'DTX3L', 'NUB1', 'IRF9', 'UBE2L6', 'TAP1', 'WARS1', 'NLRC5']
genes_stat1 = ['SREBF2', 'MSMO1', 'DHCR24', 'SQLE', 'ACAT2', 'HMGCS1', 'ERG28', 'FDFT1']
heatmap_branchAB_binned_robust(
    ad=ad,
    genes=genes_stat1,          # 你前面从 perturb-seq 取到的 STAT1 downstream genes list
    maskA=maskA, maskB=maskB,
    branch_A_dpt=branch_A_dpt, branch_B_dpt=branch_B_dpt,
    out_png=out_png,
    n_bins=50,
    min_n=20,
    layer='norm_expr_log1p',                # 或者 "log1p_norm"
    clip_q=(0.02, 0.98),
    title="SREBF2 downstream genes (Perturb-seq Stim8hr) show branch-specific dynamics"
)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency, fisher_exact

CLUSTER_KEY = "leiden_T_0.6_relabel"
RESP_KEY    = "Respond"

In [ ]:
df1 = ad.obs.loc[:, [RESP_KEY, "branch_AB_0p5"]].copy()
tab1 = pd.crosstab(df1["branch_AB_0p5"], df1[RESP_KEY], normalize="columns")  # 每个Respond列归一化
display(tab1)

# ---- color map: A=red, B=blue ----
cols = list(tab1.columns)

# 先按列名匹配（推荐）
color_map = {}
for c in cols:
    lc = str(c).lower()
    if "a" == lc or "branch_a" in lc or lc.endswith("_a") or lc.startswith("a"):
        color_map[c] = "red"
    elif "b" == lc or "branch_b" in lc or lc.endswith("_b") or lc.startswith("b"):
        color_map[c] = "blue"

# 如果没匹配上（比如列名是 0/1），就按顺序给两色
if len(color_map) != len(cols):
    fallback = ["red", "blue"]
    color_map = {c: fallback[i % 2] for i, c in enumerate(cols)}

colors = [color_map[c] for c in cols]

ax = tab1.T.plot(kind="bar", figsize=(6,4), color=colors)
ax.set_ylabel("Fraction within Respond group")
ax.set_title("Branch composition (A/B) within Responder vs Non-Responder")
plt.xticks(rotation=0)

ax.legend(title="branch_AB_0p5", loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)

p_txt = "<1e-300" if p == 0 or p < 1e-300 else f"{p:.2e}"
ax.text(0.02, 0.98, f"Chi-square: χ²={chi2:.1f}, dof={dof}, p={p_txt}",
        transform=ax.transAxes, ha="left", va="top", fontsize=10,
        bbox=dict(boxstyle="round,pad=0.25", facecolor="white", alpha=0.75, edgecolor="none"))

plt.tight_layout()
plt.savefig(os.path.join("/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/10_TF_analysis/figures",
                         "branch_composition_A_B_within_Responder_vs_Non-Responder.png"),
            dpi=300, bbox_inches="tight")
plt.show()



In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

def _get_expr_matrix(ad, genes, use_raw=True, layer=None):
    if use_raw and ad.raw is not None:
        X = ad.raw[:, genes].X
    else:
        X = ad[:, genes].layers[layer] if layer is not None else ad[:, genes].X
    X = X.toarray() if hasattr(X, "toarray") else np.asarray(X)
    return X

def _robust_zscore_cols(M):
    med = np.nanmedian(M, axis=0, keepdims=True)
    mad = np.nanmedian(np.abs(M - med), axis=0, keepdims=True) + 1e-12
    return (M - med) / (1.4826 * mad)

def _bin_means_with_cluster(dpt, X, cluster, n_bins=60, min_n=15):
    """
    dpt: (n_cells,)
    X:   (n_cells, n_genes)
    cluster: (n_cells,) (str/int)
    return:
      centers (nb,), Xm (nb, n_genes), cl_mode (nb,)  # 每个 bin 里占比最高的 cluster
    """
    dpt = np.asarray(dpt, float)
    X = np.asarray(X, float)
    cluster = np.asarray(cluster)

    m = np.isfinite(dpt)
    dpt = dpt[m]; X = X[m]; cluster = cluster[m]

    # within-branch normalize to 0-1
    d0, d1 = np.nanmin(dpt), np.nanmax(dpt)
    u = np.zeros_like(dpt) if d1 <= d0 else (dpt - d0) / (d1 - d0)

    edges = np.linspace(0.0, 1.0, n_bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])

    C, Xm, cl_mode = [], [], []
    for i in range(n_bins):
        sel = (u >= edges[i]) & (u < edges[i+1]) if i < n_bins-1 else (u >= edges[i]) & (u <= edges[i+1])
        if sel.sum() < min_n:
            continue
        Xm.append(np.nanmean(X[sel], axis=0))
        C.append(centers[i])

        # mode cluster
        vals, cnt = np.unique(cluster[sel].astype(str), return_counts=True)
        cl_mode.append(vals[np.argmax(cnt)])

    return np.array(C), np.vstack(Xm), np.array(cl_mode, dtype=object)

def _make_discrete_cmap(labels):
    # 给离散 cluster 分配颜色（不手工指定颜色，只用 matplotlib 默认循环）
    uniq = list(pd.unique(pd.Series(labels).astype(str)))
    uniq_sorted = sorted(uniq, key=lambda x: (len(x), x))  # 稍微稳定一点
    colors = [mpl.rcParams["axes.prop_cycle"].by_key()["color"][i % 10] for i in range(len(uniq_sorted))]
    cmap = mpl.colors.ListedColormap(colors)
    lut = {lab:i for i,lab in enumerate(uniq_sorted)}
    return cmap, lut, uniq_sorted

def heatmap_branchAB_vertical_with_dpt_and_cluster_bar(
    ad,
    genes,
    maskA, maskB,
    branch_A_dpt, branch_B_dpt,
    cluster_key="leiden_T_0.6_relabel",
    out_png=None,
    n_bins=60,
    min_n=20,
    use_raw=True,
    layer=None,
    clip_q=(0.02, 0.98),
    title="STAT1 downstream genes (Perturb-seq Stim8hr) show branch-specific dynamics"
):
    # gene filtering
    gene_space = ad.raw.var_names if (use_raw and ad.raw is not None) else ad.var_names
    genes_use = [g for g in genes if g in gene_space]
    if len(genes_use) < 5:
        raise ValueError("Too few genes found in adata gene space.")

    maskA = np.asarray(maskA).astype(bool)
    maskB = np.asarray(maskB).astype(bool)

    adA = ad[maskA].copy()
    adB = ad[maskB].copy()

    # cluster labels per cell
    clA = adA.obs[cluster_key].astype(str).values
    clB = adB.obs[cluster_key].astype(str).values

    # expr
    XA = _get_expr_matrix(adA, genes_use, use_raw=use_raw, layer=layer)
    XB = _get_expr_matrix(adB, genes_use, use_raw=use_raw, layer=layer)

    # bin mean + cluster mode in each bin
    cA, XmA, clA_bin = _bin_means_with_cluster(branch_A_dpt, XA, clA, n_bins=n_bins, min_n=min_n)
    cB, XmB, clB_bin = _bin_means_with_cluster(branch_B_dpt, XB, clB, n_bins=n_bins, min_n=min_n)

    # robust scale across both branches
    Xm = np.vstack([XmA, XmB])              # (binsA+binsB) x genes
    Zm = _robust_zscore_cols(Xm)
    lo, hi = np.nanquantile(Zm, clip_q[0]), np.nanquantile(Zm, clip_q[1])
    lo = float(lo); hi = float(hi)
    Zm = np.clip(Zm, lo, hi)

    ZA = Zm[:XmA.shape[0]]
    ZB = Zm[XmA.shape[0]:]

    # make cluster discrete colormap (based on both branches' bins)
    cmap_cl, lut, cl_order = _make_discrete_cmap(np.concatenate([clA_bin, clB_bin]))
    clA_code = np.array([lut[str(x)] for x in clA_bin], int)[None, :]
    clB_code = np.array([lut[str(x)] for x in clB_bin], int)[None, :]

    # ----- plotting: 2 rows (A,B), each has 2 bars + heatmap -----
    fig = plt.figure(figsize=(12.8, 0.30 * len(genes_use) + 4.4))
    gs = fig.add_gridspec(nrows=4, ncols=1, height_ratios=[0.18, 1.0, 0.18, 1.0], hspace=0.10)

    # Branch A bars
    axA_bar = fig.add_subplot(gs[0, 0])
    # stack two annotation rows: dpt row and cluster row
    barA = np.vstack([cA[None, :], clA_code])
    axA_bar.imshow(barA, aspect="auto", interpolation="nearest",
                   cmap=mpl.cm.viridis, vmin=0, vmax=1)  # dpt row ok; cluster row will be wrong in this cmap
    # workaround: draw dpt row and cluster row separately
    axA_bar.clear()
    axA_bar.imshow(cA[None, :], aspect="auto", interpolation="nearest",
                   cmap="viridis", vmin=0, vmax=1)
    axA_bar.imshow(clA_code, aspect="auto", interpolation="nearest",
                   cmap=cmap_cl, vmin=-0.5, vmax=len(cl_order)-0.5,
                   extent=[-0.5, cA.shape[0]-0.5, 0.5, 1.5])  # put on second row
    axA_bar.set_yticks([0, 1])
    axA_bar.set_yticklabels(["dpt", "cluster"], fontsize=10)
    axA_bar.set_xticks([])
    axA_bar.set_title(title, fontsize=13, pad=6)

    # Branch A heatmap
    axA = fig.add_subplot(gs[1, 0])
    imA = axA.imshow(ZA.T, aspect="auto", interpolation="nearest", cmap="RdBu_r", vmin=lo, vmax=hi)
    axA.set_yticks(np.arange(len(genes_use)))
    axA.set_yticklabels(genes_use, fontsize=10)
    axA.set_xticks([ZA.shape[0]/2])
    axA.set_xticklabels([f"Branch A (binned, {ZA.shape[0]} bins)"], fontsize=11)
    axA.tick_params(axis="x", bottom=False, top=False)
    axA.spines["top"].set_visible(False)
    axA.spines["right"].set_visible(False)

    # Branch B bars
    axB_bar = fig.add_subplot(gs[2, 0])
    axB_bar.imshow(cB[None, :], aspect="auto", interpolation="nearest", cmap="viridis", vmin=0, vmax=1)
    axB_bar.imshow(clB_code, aspect="auto", interpolation="nearest",
                   cmap=cmap_cl, vmin=-0.5, vmax=len(cl_order)-0.5,
                   extent=[-0.5, cB.shape[0]-0.5, 0.5, 1.5])
    axB_bar.set_yticks([0, 1])
    axB_bar.set_yticklabels(["dpt", "cluster"], fontsize=10)
    axB_bar.set_xticks([])
    axB_bar.set_title("")  # no title

    # Branch B heatmap
    axB = fig.add_subplot(gs[3, 0], sharey=axA)
    imB = axB.imshow(ZB.T, aspect="auto", interpolation="nearest", cmap="RdBu_r", vmin=lo, vmax=hi)
    axB.set_yticks(np.arange(len(genes_use)))
    axB.set_yticklabels(genes_use, fontsize=10)
    axB.set_xticks([ZB.shape[0]/2])
    axB.set_xticklabels([f"Branch B (binned, {ZB.shape[0]} bins)"], fontsize=11)
    axB.tick_params(axis="x", bottom=False, top=False)
    axB.spines["top"].set_visible(False)
    axB.spines["right"].set_visible(False)

    # colorbars (right side)
    # dpt colorbar
    cax_dpt = fig.add_axes([0.92, 0.84, 0.015, 0.10])
    cb_dpt = plt.colorbar(mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(0,1), cmap="viridis"), cax=cax_dpt)
    cb_dpt.set_label("within-branch dpt (0–1)", rotation=90)

    # expression colorbar
    cax_exp = fig.add_axes([0.92, 0.20, 0.015, 0.58])
    cb_exp = plt.colorbar(imA, cax=cax_exp)
    cb_exp.set_label("Expression (robust z, clipped)", rotation=90)

    # cluster legend as text (compact)
    # 放在右侧空白处
    ax_leg = fig.add_axes([0.94, 0.02, 0.05, 0.15])
    ax_leg.axis("off")
    ax_leg.text(0.0, 1.0, "cluster", fontsize=10, va="top")
    for i, lab in enumerate(cl_order[:10]):  # 如果 cluster 多于10个也够用了
        ax_leg.text(0.0, 0.85 - i*0.08, lab, color=cmap_cl(i), fontsize=9, va="top")

    plt.tight_layout(rect=[0, 0, 0.90, 1])

    if out_png is not None:
        os.makedirs(os.path.dirname(out_png), exist_ok=True)
        fig.savefig(out_png, dpi=300, bbox_inches="tight")
        print("[DONE] saved:", out_png)

    plt.show()
    plt.close(fig)

out_png = "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/10_TF_analysis/figures/Fig_STAT1_downstream_branchAB_vertical_dpt_cluster.png"

heatmap_branchAB_vertical_with_dpt_and_cluster_bar(
    ad=ad,
    genes=genes_stat1,
    maskA=maskA, maskB=maskB,
    branch_A_dpt=branch_A_dpt, branch_B_dpt=branch_B_dpt,
    cluster_key="leiden_T_0.6_relabel",
    out_png=out_png,
    n_bins=60,
    min_n=20,
    use_raw=(ad.raw is not None),
    layer=None,          # 或你的 log1p layer
    clip_q=(0.02, 0.98),
    title="STAT1 downstream genes (Perturb-seq Stim8hr) show branch-specific dynamics"
)

In [ ]:
def to_uniform_01(x: np.ndarray) -> np.ndarray:
    """Rank-based mapping to ~Uniform(0,1)."""
    x = np.asarray(x, dtype=float)
    n = x.size
    if n <= 1:
        return np.zeros_like(x)
    r = rankdata(x, method="average")  # 1..n
    return (r - 1) / (n - 1)

branch_A_dpt_u = to_uniform_01(branch_A_dpt)
branch_B_dpt_u = to_uniform_01(branch_B_dpt)

print(branch_A_dpt_u.min(), branch_A_dpt_u.max(), " | ", branch_B_dpt_u.min(), branch_B_dpt_u.max())

In [ ]:
ad.obs['branch_preference_A_sigmoid'].value_counts()

In [ ]:
branch_A_naive = ad.obs.loc[maskA, NAIVE_KEY].astype(float).values
branch_B_naive = ad.obs.loc[maskB, NAIVE_KEY].astype(float).values

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

def binned_mean_curve(x, y, n_bins=60, min_n=15):
    x = np.asarray(x, float); y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    bins = np.linspace(0.0, 1.0, n_bins + 1)

    xc, yc = [], []
    for i in range(n_bins):
        sel = (x >= bins[i]) & (x < bins[i+1]) if i < n_bins-1 else (x >= bins[i]) & (x <= bins[i+1])
        if sel.sum() < min_n:
            continue
        xc.append(0.5 * (bins[i] + bins[i+1]))
        yc.append(np.mean(y[sel]))
    return np.array(xc), np.array(yc)

def smooth_rolling(y, win=9):
    # win 必须是奇数更好看；win 越大越平滑（建议 9~21）
    s = pd.Series(y)
    return s.rolling(window=win, center=True, min_periods=max(3, win//3)).mean().to_numpy()

def plot_two_lines(xA, yA, xB, yB, title, ylabel, win=11):
    yA_s = smooth_rolling(yA, win=win)
    yB_s = smooth_rolling(yB, win=win)

    plt.figure(figsize=(6.2, 4.8))
    plt.plot(xA, yA_s, linewidth=2.8, color="red",  label="A")
    plt.plot(xB, yB_s, linewidth=2.8, color="blue", label="B")
    plt.xlabel("within-branch uniform pseudotime (0-1)")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend(frameon=False)
    plt.tight_layout()
    #plt.savefig(os.path.join(out_dir, f"{title}.png"), dpi=200, bbox_inches="tight")
    plt.show()

# 例子：naive
xA, yA = binned_mean_curve(branch_A_dpt_u, branch_A_naive, n_bins=70, min_n=20)
xB, yB = binned_mean_curve(branch_B_dpt_u, branch_B_naive, n_bins=70, min_n=20)
plot_two_lines(xA, yA, xB, yB, "Naive score trend (smoothed)", "score_Naive", win=13)



In [ ]:
import numpy as np
import pandas as pd

# 1) 取出矩阵 + 列名
X = ad.obsm["X_regulon_auc"]
cols = ad.uns["regulon_auc_cols"]

# 2) 保险检查
if not isinstance(X, np.ndarray):
    raise TypeError(f"ad.obsm['X_regulon_auc'] 不是 ndarray，而是 {type(X)}")
if len(cols) != X.shape[1]:
    raise ValueError(f"列名数量 len(cols)={len(cols)} != X.shape[1]={X.shape[1]}")
if X.shape[0] != ad.n_obs:
    raise ValueError(f"X.shape[0]={X.shape[0]} != ad.n_obs={ad.n_obs}")

# 3) 转成 DataFrame 并放回去
ad.obsm["X_regulon_auc"] = pd.DataFrame(X, index=ad.obs_names, columns=pd.Index(cols, name="regulon"))

print("[OK] Converted ad.obsm['X_regulon_auc'] to DataFrame:", ad.obsm["X_regulon_auc"].shape)
print("First 5 cols:", ad.obsm["X_regulon_auc"].columns[:5].tolist())

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# -----------------------------
# 1) 你的函数：分箱均值 + 平滑 + 画两条线
# -----------------------------
def binned_mean_curve_with_bootci(x, y, n_bins=60, min_n=15, n_boot=300, ci=0.95, seed=0):
    """
    返回：
      xc: bin center
      y_mean: 每个 bin 的均值
      y_lo, y_hi: bootstrap (ci)置信区间
      n_in_bin: 每个 bin 样本数（可用于诊断）
    """
    rng = np.random.default_rng(seed)

    x = np.asarray(x, float)
    y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    alpha = 1.0 - ci
    q_lo, q_hi = alpha / 2.0, 1.0 - alpha / 2.0

    xc, y_mean, y_lo, y_hi, n_in_bin = [], [], [], [], []

    for i in range(n_bins):
        if i < n_bins - 1:
            sel = (x >= bins[i]) & (x < bins[i + 1])
        else:
            sel = (x >= bins[i]) & (x <= bins[i + 1])

        n = int(sel.sum())
        if n < min_n:
            continue

        ys = y[sel]
        mu = float(np.mean(ys))

        # bootstrap: resample ys with replacement, compute mean
        # n_boot 不要太大（200~1000够），避免慢
        idx = rng.integers(0, n, size=(n_boot, n))
        boot_means = ys[idx].mean(axis=1)

        lo = float(np.quantile(boot_means, q_lo))
        hi = float(np.quantile(boot_means, q_hi))

        xc.append(0.5 * (bins[i] + bins[i + 1]))
        y_mean.append(mu)
        y_lo.append(lo)
        y_hi.append(hi)
        n_in_bin.append(n)

    return (np.array(xc), np.array(y_mean), np.array(y_lo), np.array(y_hi), np.array(n_in_bin))


def smooth_rolling(y, win=9):
    s = pd.Series(y)
    return s.rolling(window=win, center=True, min_periods=max(3, win//3)).mean().to_numpy()


def plot_two_lines_with_ci(
    xA, yA, loA, hiA,
    xB, yB, loB, hiB,
    title, ylabel,
    win=11,
    colorA="red", colorB="blue",
    fillA=(1.0, 0.6, 0.6),  # 浅红（RGB）
    fillB=(0.6, 0.7, 1.0),  # 浅蓝（RGB）
    fill_alpha=0.28,
    show=True,
    savepath=None
):
    # 平滑：均值线平滑；CI带也一起平滑（更“顺眼”）
    yA_s  = smooth_rolling(yA,  win=win)
    loA_s = smooth_rolling(loA, win=win)
    hiA_s = smooth_rolling(hiA, win=win)

    yB_s  = smooth_rolling(yB,  win=win)
    loB_s = smooth_rolling(loB, win=win)
    hiB_s = smooth_rolling(hiB, win=win)

    fig = plt.figure(figsize=(6.2, 4.8))

    # CI band
    plt.fill_between(xA, loA_s, hiA_s, color=fillA, alpha=fill_alpha, linewidth=0)
    plt.fill_between(xB, loB_s, hiB_s, color=fillB, alpha=fill_alpha, linewidth=0)

    # mean line
    plt.plot(xA, yA_s, linewidth=2.8, color=colorA, label="A")
    plt.plot(xB, yB_s, linewidth=2.8, color=colorB, label="B")

    plt.xlabel("within-branch uniform pseudotime (0-1)")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend(frameon=False)
    plt.tight_layout()

    if savepath is not None:
        fig.savefig(savepath, dpi=200, bbox_inches="tight")

    if show:
        plt.show()
    plt.close(fig)

def binned_mean_curve(x, y, n_bins=60, min_n=15):
    x = np.asarray(x, float); y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    bins = np.linspace(0.0, 1.0, n_bins + 1)

    xc, yc = [], []
    for i in range(n_bins):
        if i < n_bins - 1:
            sel = (x >= bins[i]) & (x < bins[i + 1])
        else:
            sel = (x >= bins[i]) & (x <= bins[i + 1])
        if sel.sum() < min_n:
            continue
        xc.append(0.5 * (bins[i] + bins[i + 1]))
        yc.append(np.mean(y[sel]))
    return np.array(xc), np.array(yc)

def smooth_rolling(y, win=9):
    s = pd.Series(y)
    return s.rolling(window=win, center=True, min_periods=max(3, win // 3)).mean().to_numpy()

def plot_two_lines(xA, yA, xB, yB, title, ylabel, win=11, show=False, savepath=None):
    yA_s = smooth_rolling(yA, win=win)
    yB_s = smooth_rolling(yB, win=win)

    fig = plt.figure(figsize=(6.2, 4.8))
    plt.plot(xA, yA_s, linewidth=2.8, color="red",  label="A")
    plt.plot(xB, yB_s, linewidth=2.8, color="blue", label="B")
    plt.xlabel("within-branch uniform pseudotime (0-1)")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend(frameon=False)
    plt.tight_layout()

    if savepath is not None:
        fig.savefig(savepath, dpi=200, bbox_inches="tight")
    if show:
        plt.show()
    plt.close(fig)
    return fig

def safe_fname(s: str, max_len=140):
    s = re.sub(r"[\/\\\:\*\?\"\<\>\|]+", "_", str(s))
    s = re.sub(r"\s+", " ", s).strip()
    if len(s) > max_len:
        s = s[:max_len].rstrip()
    return s


# -----------------------------
# 2) 批量绘图主函数
# -----------------------------
def plot_all_regulons_trends(
    ad,
    maskA,
    maskB,
    branch_A_dpt_u,
    branch_B_dpt_u,
    out_dir,
    n_bins=70,
    min_n=20,
    win=13,
    ylabel_prefix="regulon_AUC",
    make_pdf=True,
    pdf_name="regulon_trends_AB.pdf",
    show_first_n=0,
    tf_subset=None,
    min_points_per_branch=60,
    n_boot=400,
    ci=0.95,
    seedA=0,
    seedB=1,
    fill_alpha=0.28,
    fillA=(1.0, 0.6, 0.6),  # 浅红
    fillB=(0.6, 0.7, 1.0),  # 浅蓝
):
    os.makedirs(out_dir, exist_ok=True)

    X = ad.obsm["X_regulon_auc"]
    if not isinstance(X, pd.DataFrame):
        raise TypeError("ad.obsm['X_regulon_auc'] 必须是 pandas.DataFrame（cells x TF）")

    # masks
    if isinstance(maskA, pd.Series):
        maskA = maskA.values
    if isinstance(maskB, pd.Series):
        maskB = maskB.values
    maskA = np.asarray(maskA).astype(bool)
    maskB = np.asarray(maskB).astype(bool)

    branch_A_dpt_u = np.asarray(branch_A_dpt_u, float)
    branch_B_dpt_u = np.asarray(branch_B_dpt_u, float)

    if maskA.shape[0] != ad.n_obs or maskB.shape[0] != ad.n_obs:
        raise ValueError("maskA/maskB 长度必须等于 ad.n_obs")
    if branch_A_dpt_u.shape[0] != maskA.sum():
        raise ValueError("branch_A_dpt_u 长度必须等于 maskA.sum()")
    if branch_B_dpt_u.shape[0] != maskB.sum():
        raise ValueError("branch_B_dpt_u 长度必须等于 maskB.sum()")

    # TF list
    all_tfs = list(X.columns)
    if tf_subset is None:
        tfs = all_tfs
    else:
        missing = [t for t in tf_subset if t not in X.columns]
        if missing:
            raise ValueError(f"tf_subset 里有 TF 不在 X_regulon_auc 列中：{missing[:10]}")
        tfs = list(tf_subset)

    # output
    pdf_path = os.path.join(out_dir, pdf_name)
    pdf = PdfPages(pdf_path) if make_pdf else None

    summary = []
    shown = 0

    XA = X.loc[maskA]  # branch A cells
    XB = X.loc[maskB]  # branch B cells

    for tf in tfs:
        yA_raw = XA[tf].astype(float).to_numpy()
        yB_raw = XB[tf].astype(float).to_numpy()

        okA = np.isfinite(branch_A_dpt_u) & np.isfinite(yA_raw)
        okB = np.isfinite(branch_B_dpt_u) & np.isfinite(yB_raw)
        nA = int(okA.sum())
        nB = int(okB.sum())

        if nA < min_points_per_branch or nB < min_points_per_branch:
            summary.append((tf, nA, nB, "SKIP_low_points"))
            continue

        # --- compute binned mean + bootstrap CI ---
        xA, yA, loA, hiA, nbinA = binned_mean_curve_with_bootci(
            branch_A_dpt_u[okA], yA_raw[okA],
            n_bins=n_bins, min_n=min_n, n_boot=n_boot, ci=ci, seed=seedA
        )
        xB, yB, loB, hiB, nbinB = binned_mean_curve_with_bootci(
            branch_B_dpt_u[okB], yB_raw[okB],
            n_bins=n_bins, min_n=min_n, n_boot=n_boot, ci=ci, seed=seedB
        )

        if len(xA) < 6 or len(xB) < 6:
            summary.append((tf, nA, nB, "SKIP_few_bins"))
            continue

        title = f"{tf} trend (A vs B)"
        ylabel = f"{ylabel_prefix}: {tf}"

        png_name = safe_fname(f"regulon_{tf}_trend_AB_CI.png")
        png_path = os.path.join(out_dir, png_name)

        show_now = (shown < show_first_n)

        # --- draw + save PNG ---
        plot_two_lines_with_ci(
            xA, yA, loA, hiA,
            xB, yB, loB, hiB,
            title=title,
            ylabel=ylabel,
            win=win,
            fillA=fillA,
            fillB=fillB,
            fill_alpha=fill_alpha,
            show=show_now,
            savepath=png_path
        )

        # --- write to PDF (also with CI) ---
        if pdf is not None:
            fig = plt.figure(figsize=(6.2, 4.8))

            # smooth for pdf too
            yA_s  = smooth_rolling(yA,  win=win)
            loA_s = smooth_rolling(loA, win=win)
            hiA_s = smooth_rolling(hiA, win=win)
            yB_s  = smooth_rolling(yB,  win=win)
            loB_s = smooth_rolling(loB, win=win)
            hiB_s = smooth_rolling(hiB, win=win)

            plt.fill_between(xA, loA_s, hiA_s, color=fillA, alpha=fill_alpha, linewidth=0)
            plt.fill_between(xB, loB_s, hiB_s, color=fillB, alpha=fill_alpha, linewidth=0)
            plt.plot(xA, yA_s, linewidth=2.8, color="red",  label="A")
            plt.plot(xB, yB_s, linewidth=2.8, color="blue", label="B")

            plt.xlabel("within-branch uniform pseudotime (0-1)")
            plt.ylabel(ylabel)
            plt.title(title)
            plt.legend(frameon=False)
            plt.tight_layout()
            pdf.savefig(fig, dpi=200)
            plt.close(fig)

        summary.append((tf, nA, nB, "OK"))
        if show_now:
            shown += 1

    if pdf is not None:
        pdf.close()

    summary_df = pd.DataFrame(summary, columns=["TF", "nA_valid", "nB_valid", "status"])
    summary_df.to_csv(os.path.join(out_dir, "regulon_trend_summary.csv"), index=False)

    print(f"[DONE] out_dir = {out_dir}")
    if make_pdf:
        print(f"[DONE] PDF = {pdf_path}")
    print("[DONE] summary CSV saved: regulon_trend_summary.csv")
    try:
        display(summary_df["status"].value_counts())
    except Exception:
        print(summary_df["status"].value_counts())

    return summary_df


# -----------------------------
# 3) 你只需要改这里：输出目录 + 跑起来
# -----------------------------
out_dir = "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/10_TF_analysis/figures/"
summary_df = plot_all_regulons_trends(
    ad=ad,
    maskA=maskA,
    maskB=maskB,
    branch_A_dpt_u=branch_A_dpt_u,
    branch_B_dpt_u=branch_B_dpt_u,
    out_dir=out_dir,
    n_bins=70,
    min_n=20,
    win=13,
    make_pdf=True,
    pdf_name="regulon_trends_AB.pdf",
    show_first_n=0,          # 想看前 5 个就设 5
    tf_subset=None,          # 或者传 list：["TCF7", "BATF", ...]
    min_points_per_branch=60
)

In [ ]:
len(ad.uns['regulon_auc_cols'])

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def zscore_cols(df: pd.DataFrame) -> pd.DataFrame:
    X = df.astype(float)
    mu = X.mean(axis=0)
    sd = X.std(axis=0, ddof=0).replace(0, np.nan)
    return (X - mu) / sd

def sort_rows_by_hclust(mat: pd.DataFrame, method="average", metric="correlation"):
    """按行聚类排序（如果没装scipy就原样返回）"""
    try:
        from scipy.cluster.hierarchy import linkage, leaves_list
        from scipy.spatial.distance import pdist
    except Exception:
        return mat, None

    X = mat.values
    # correlation distance: 1 - corr
    if metric == "correlation":
        d = pdist(X, metric="correlation")
    else:
        d = pdist(X, metric=metric)
    Z = linkage(d, method=method)
    order = leaves_list(Z)
    return mat.iloc[order], order

def make_scenic_cluster_heatmap(
    ad,
    cluster_col,                 # e.g. "leiden_T_0.6_relabel"
    out_dir,
    fname_prefix="Fig1_SCENIC_cluster_heatmap",
    top_n=80,                    # 只展示变化最大的TF数
    min_cells_per_cluster=50,
    zscore=True,                 # 强烈建议 True
    cluster_order=None,          # ["C1","C2",...], None则按出现顺序
    cluster_as_str=True,
    hclust_rows=True,            # 行聚类让结构更明显
    cmap="RdBu_r",
    vlim=2.5
):
    os.makedirs(out_dir, exist_ok=True)

    X = ad.obsm["X_regulon_auc"]
    if not isinstance(X, pd.DataFrame):
        raise TypeError("ad.obsm['X_regulon_auc'] 必须是 DataFrame (cells x regulons).")

    cl = ad.obs[cluster_col]
    cl = cl.astype(str) if cluster_as_str else cl

    # 过滤掉太小的cluster（图会更稳定）
    vc = cl.value_counts()
    keep_clusters = vc[vc >= min_cells_per_cluster].index.tolist()
    mask_keep = cl.isin(keep_clusters).values

    Xk = X.loc[mask_keep].copy()
    clk = cl.loc[mask_keep].copy()

    # cluster顺序
    if cluster_order is None:
        cluster_order = [c for c in keep_clusters]
    else:
        cluster_order = [c for c in cluster_order if c in keep_clusters]

    # 先对每个TF做z-score（跨所有细胞）
    if zscore:
        Xproc = zscore_cols(Xk)
    else:
        Xproc = Xk.astype(float)

    # 每个cluster取均值 -> TF x cluster
    mat = (
        Xproc.assign(_cluster=clk.values)
             .groupby("_cluster")
             .mean(numeric_only=True)
             .T
    )
    mat = mat[cluster_order]  # reorder columns

    # 选择跨cluster变化最大的 top_n TF
    var = mat.var(axis=1, ddof=0)
    tf_use = var.sort_values(ascending=False).head(top_n).index.tolist()
    mat_top = mat.loc[tf_use].copy()

    # 行聚类排序（可选）
    if hclust_rows:
        mat_plot, order = sort_rows_by_hclust(mat_top, method="average", metric="correlation")
    else:
        mat_plot, order = mat_top, None

    # 保存矩阵
    csv_path = os.path.join(out_dir, f"{fname_prefix}.matrix_top{top_n}.csv")
    mat_plot.to_csv(csv_path)

    # 画图
    fig_h = max(5.2, 0.28 * mat_plot.shape[0] + 1.6)
    fig_w = max(6.2, 1.0 * mat_plot.shape[1] + 3.2)
    fig = plt.figure(figsize=(fig_w, fig_h))
    ax = plt.gca()

    im = ax.imshow(mat_plot.values, aspect="auto", cmap=cmap, vmin=-vlim, vmax=vlim)

    ax.set_yticks(np.arange(mat_plot.shape[0]))
    ax.set_yticklabels(mat_plot.index.tolist(), fontsize=9)

    ax.set_xticks(np.arange(mat_plot.shape[1]))
    # 在列名里加上n
    ns = clk.value_counts().to_dict()
    xt = [f"{c}\n(n={ns.get(c,0)})" for c in mat_plot.columns.tolist()]
    ax.set_xticklabels(xt, fontsize=10)

    title = f"SCENIC regulon activity differs across clusters (top {top_n} variable regulons)"
    ax.set_title(title, fontsize=13, pad=10)

    cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cbar.set_label("Mean regulon AUC (z-score across cells)" if zscore else "Mean regulon AUC", rotation=90)

    plt.tight_layout()

    png_path = os.path.join(out_dir, f"{fname_prefix}.top{top_n}.png")
    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    print("[DONE] saved heatmap:", png_path)
    print("[DONE] saved matrix:", csv_path)
    print("[INFO] clusters kept:", cluster_order)
    return mat_plot


# -----------------------------
# 用法：你只要改 cluster_col 和输出目录
# -----------------------------
out_dir = "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/10_TF_analysis/figures/"
mat_plot = make_scenic_cluster_heatmap(
    ad=ad,
    cluster_col="leiden_T_0.6_relabel",   # <- 改成你的cluster列名
    out_dir=out_dir,
    fname_prefix="Fig1_SCENIC_cluster_heatmap",
    top_n=80,
    min_cells_per_cluster=50,
    zscore=True,
    cluster_order=['1', '2', '3', '4'],  # 你有就填，没有就删掉这行
    hclust_rows=True,
    vlim=2.5
)


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def zscore_cols(df: pd.DataFrame) -> pd.DataFrame:
    X = df.astype(float)
    mu = X.mean(axis=0)
    sd = X.std(axis=0, ddof=0).replace(0, np.nan)
    return (X - mu) / sd

def pick_representative_tfs_per_cluster(
    ad,
    cluster_col,
    k=8,                 # 每个cluster选k个（5~10）
    z_min=0.6,           # 至少在该cluster里平均zscore>z_min
    delta_min=0.3,       # 该cluster比第二高cluster至少高delta_min
    min_cells=50,
    cluster_order=None
):
    X = ad.obsm["X_regulon_auc"]
    if not isinstance(X, pd.DataFrame):
        raise TypeError("ad.obsm['X_regulon_auc'] 必须是 DataFrame (cells x TF).")

    cl = ad.obs[cluster_col].astype(str)
    vc = cl.value_counts()
    keep = vc[vc >= min_cells].index.tolist()
    m = cl.isin(keep).values

    Xk = X.loc[m].copy()
    clk = cl.loc[m].copy()

    if cluster_order is None:
        cluster_order = keep
    else:
        cluster_order = [c for c in cluster_order if c in keep]

    # z-score across all cells
    Xz = zscore_cols(Xk)

    # cluster mean zscore matrix: TF x cluster
    mean_z = (
        Xz.assign(_cluster=clk.values)
          .groupby("_cluster")
          .mean(numeric_only=True)
          .T
    )[cluster_order]

    # 每个 TF 计算 “top1 - top2” 用于过滤“泛高TF”
    sorted_vals = np.sort(mean_z.values, axis=1)  # ascending
    top1 = sorted_vals[:, -1]
    top2 = sorted_vals[:, -2] if mean_z.shape[1] >= 2 else np.full_like(top1, np.nan)
    gap = top1 - top2
    gap = pd.Series(gap, index=mean_z.index, name="top1_minus_top2")

    picked = {}
    picked_union = []

    for c in cluster_order:
        s = mean_z[c].copy()

        # 过滤：强度 + 特异性
        s = s[s >= z_min]
        s = s[gap.loc[s.index] >= delta_min]

        # 如果过滤后不够k，就放宽（保证每个cluster都有代表TF）
        if s.shape[0] < k:
            s = mean_z[c].copy().sort_values(ascending=False)
        else:
            s = s.sort_values(ascending=False)

        top = s.head(k).index.tolist()
        picked[c] = top
        picked_union.extend(top)

    picked_union = list(dict.fromkeys(picked_union))  # 保序去重
    return mean_z, picked, picked_union

def plot_heatmap(mat, out_png, title, cmap="RdBu_r", vlim=2.5):
    fig_h = max(5.0, 0.30 * mat.shape[0] + 1.6)
    fig_w = max(6.0, 1.0 * mat.shape[1] + 2.8)
    fig = plt.figure(figsize=(fig_w, fig_h))
    ax = plt.gca()
    im = ax.imshow(mat.values, aspect="auto", cmap=cmap, vmin=-vlim, vmax=vlim)

    ax.set_yticks(np.arange(mat.shape[0]))
    ax.set_yticklabels(mat.index.tolist(), fontsize=9)

    ax.set_xticks(np.arange(mat.shape[1]))
    ax.set_xticklabels(mat.columns.tolist(), fontsize=10)

    ax.set_title(title, fontsize=13, pad=10)
    cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cbar.set_label("Mean regulon AUC (z-score across cells)", rotation=90)

    plt.tight_layout()
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

# -----------------------------
# 运行：每个cluster挑 8 个TF（总量一般在30~60之间）
# -----------------------------
cluster_col = "leiden_T_0.6_relabel"   # 改成你的
cluster_order = ["1", "2", "3", "4"]  # 有就填

mean_z, picked, picked_union = pick_representative_tfs_per_cluster(
    ad,
    cluster_col=cluster_col,
    k=8,          # 5~10
    z_min=0.6,
    delta_min=0.3,
    min_cells=50,
    cluster_order=cluster_order
)

out_dir = "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/10_TF_analysis/figures/"
os.makedirs(out_dir, exist_ok=True)

# 保存每个cluster的top TF列表
rows = []
for c, tfs in picked.items():
    for rank, tf in enumerate(tfs, 1):
        rows.append({"cluster": c, "rank": rank, "TF": tf, "mean_z": float(mean_z.loc[tf, c])})
picked_df = pd.DataFrame(rows)
picked_df.to_csv(os.path.join(out_dir, "Fig1_cluster_topTFs.csv"), index=False)

# 用“union集合”画一个不太大的 heatmap
mat_fig1 = mean_z.loc[picked_union, :]
mat_fig1.to_csv(os.path.join(out_dir, "Fig1_cluster_heatmap_matrix.csv"))

plot_heatmap(
    mat_fig1,
    out_png=os.path.join(out_dir, "Fig1_SCENIC_cluster_heatmap_representativeTFs.png"),
    title=f"SCENIC regulon activity differs across clusters (representative {len(picked_union)} regulons)",
    vlim=2.5
)

print("[DONE] representative TFs total:", len(picked_union))
print("[DONE] saved:", os.path.join(out_dir, "Fig1_cluster_topTFs.csv"))


In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

def resolve_tf_to_cols(X_df, tf, prefer_sign="+"):
    cols = list(map(str, X_df.columns))

    patt = re.compile(rf"^{re.escape(tf)}\s*\(([+-])\)\s*$")
    hits = [(c, patt.match(c).group(1)) for c in cols if patt.match(c)]
    if hits:
        pref = [c for c, s in hits if s == prefer_sign]
        return pref if pref else [c for c, s in hits]

    hits2 = [c for c in cols if re.search(rf"\b{re.escape(tf)}\b", c)]
    if hits2:
        plus = [c for c in hits2 if "(+)" in c]
        return plus if plus else hits2

    return []

def boxplot_two_groups_rawlabels(y, group, title, ylabel, outpath):
    df = pd.DataFrame({"y": y, "group": group})
    df = df[np.isfinite(df["y"].values)]

    g1 = df.loc[df["group"]=="Non-Responder", "y"].values
    g2 = df.loc[df["group"]=="Responder",     "y"].values

    if len(g1) < 3 or len(g2) < 3:
        return {"n_NR": len(g1), "n_R": len(g2), "p": np.nan, "status": "SKIP_empty_group"}

    try:
        p = mannwhitneyu(g1, g2, alternative="two-sided").pvalue
    except Exception:
        p = np.nan

    fig = plt.figure(figsize=(4.6, 4.8))
    plt.boxplot([g1, g2],
                labels=[f"Non-Responder\n(n={len(g1)})", f"Responder\n(n={len(g2)})"],
                showfliers=False)
    plt.ylabel(ylabel)
    plt.title(f"{title}\nMann-Whitney p={p:.2e}" if np.isfinite(p) else title)
    plt.tight_layout()
    fig.savefig(outpath, dpi=200, bbox_inches="tight")
    plt.close(fig)

    return {"n_NR": len(g1), "n_R": len(g2), "p": p, "status": "OK"}

def plot_singleTF_boxplots_in_C3_png(
    ad,
    mask_c3,
    group_col="Respond",
    tf_list=None,
    out_dir=".",
    prefer_sign="+",
    ylim=None,                  # 例如 (-2, 2)；不想固定就 None
):
    os.makedirs(out_dir, exist_ok=True)

    X = ad.obsm["X_regulon_auc"]
    if not isinstance(X, pd.DataFrame):
        raise TypeError("ad.obsm['X_regulon_auc'] 必须是 DataFrame")

    if isinstance(mask_c3, pd.Series):
        mask_c3 = mask_c3.values
    mask_c3 = np.asarray(mask_c3, bool)

    X_c3 = X.loc[mask_c3].copy()
    group = ad.obs.loc[mask_c3, group_col].astype(str).values

    # 默认 TF 列表：从 TF(+)/TF(-) 里提取基名
    if tf_list is None:
        tf_pat = re.compile(r"^([A-Za-z0-9_.-]+)\s*\(([+-])\)\s*$")
        base = []
        for c in X_c3.columns:
            m = tf_pat.match(str(c))
            if m:
                base.append(m.group(1))
        tf_list = sorted(set(base)) if len(base) else list(X_c3.columns)

    summary = []
    for tf in tf_list:
        cols = resolve_tf_to_cols(X_c3, tf, prefer_sign=prefer_sign)
        if len(cols) == 0:
            summary.append({"TF": tf, "used_col": "", "status": "SKIP_no_match", "n_NR": 0, "n_R": 0, "p": np.nan})
            continue

        # 命中多个列就取均值作为该 TF 的 AUC（你也可以改成只取 cols[0]）
        y = X_c3[cols].astype(float).mean(axis=1).to_numpy()

        out_png = os.path.join(out_dir, f"C3_boxplot_{tf}.png")

        # 画图（可选统一 ylim）
        res = boxplot_two_groups_rawlabels(
            y, group,
            title=f"C3 {tf} regulon activity (Non-Responder vs Responder)",
            ylabel=f"regulon AUC: {tf}",
            outpath=out_png
        )

        # 如果要统一 y 轴范围，可以在这里重新画一次（更简单：我给你一个开关）
        if ylim is not None and res["status"] == "OK":
            # 重新打开保存一次，固定 ylim（不显示）
            df = pd.DataFrame({"y": y, "group": group})
            df = df[np.isfinite(df["y"].values)]
            g1 = df.loc[df["group"]=="Non-Responder", "y"].values
            g2 = df.loc[df["group"]=="Responder",     "y"].values
            fig = plt.figure(figsize=(4.6, 4.8))
            plt.boxplot([g1, g2],
                        labels=[f"Non-Responder\n(n={len(g1)})", f"Responder\n(n={len(g2)})"],
                        showfliers=False)
            plt.ylim(*ylim)
            plt.ylabel(f"regulon AUC: {tf}")
            plt.title(f"C3 {tf} regulon activity (Non-Responder vs Responder)")
            plt.tight_layout()
            fig.savefig(out_png, dpi=200, bbox_inches="tight")
            plt.close(fig)

        summary.append({"TF": tf, "used_col": ";".join(cols), **res, "png": out_png})

    summary_df = pd.DataFrame(summary).sort_values(["status", "p"], ascending=[True, True])
    summary_df.to_csv(os.path.join(out_dir, "C3_TF_boxplot_summary.csv"), index=False)
    print("[DONE] saved PNGs to:", out_dir)
    print("[DONE] saved summary:", os.path.join(out_dir, "C3_TF_boxplot_summary.csv"))
    return summary_df


In [ ]:
mask_c3 = ad.obs["leiden_T_0.6_relabel"].astype(str).isin(["3", "C3"])
out_dir = "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/10_TF_analysis/figures/C3_TF_boxplots/"

tf_list = ["EOMES","PRDM1","STAT1","NFATC3","RUNX1","ARNT","PBX3","BATF","AHR","FOXO3","RELB","NFKB1","NFATC1"]

summary_df = plot_singleTF_boxplots_in_C3_png(
    ad=ad,
    mask_c3=mask_c3,
    group_col="Respond",   # 你的原始列：Non-Responder / Responder
    tf_list=tf_list,
    out_dir=out_dir,
    prefer_sign="+",
    ylim=None              # 需要统一y轴就设成例如 (-2, 2)
)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

def cliffs_delta(x, y):
    # effect size: Cliff's delta in [-1, 1]
    x = np.asarray(x); y = np.asarray(y)
    # 可能较慢但你这里 n~几千可接受
    gt = np.sum(x[:, None] > y[None, :])
    lt = np.sum(x[:, None] < y[None, :])
    return (gt - lt) / (x.size * y.size + 1e-12)

def boxplot_pretty_two_groups(y, group, tf_label, outpath=None, title_prefix="C3 regulon activity",
                              show_points=250,  # 每组最多画多少个点（防止太密）
                              point_alpha=0.18, point_size=10,
                              ylim=None):
    df = pd.DataFrame({"y": y, "group": group})
    df = df[np.isfinite(df["y"].values)]

    gNR = df.loc[df["group"]=="Non-Responder", "y"].values
    gR  = df.loc[df["group"]=="Responder",     "y"].values

    if len(gNR) < 3 or len(gR) < 3:
        return {"n_NR": len(gNR), "n_R": len(gR), "p": np.nan, "delta": np.nan, "status": "SKIP_empty_group"}

    # stats
    p = mannwhitneyu(gNR, gR, alternative="two-sided").pvalue
    d = cliffs_delta(gNR, gR)
    medNR, medR = np.median(gNR), np.median(gR)

    fig, ax = plt.subplots(figsize=(5.4, 4.8))

    # boxplot
    bp = ax.boxplot(
        [gNR, gR],
        positions=[1, 2],
        widths=0.55,
        showfliers=False,
        patch_artist=True,
        medianprops=dict(linewidth=2.2),
        boxprops=dict(linewidth=1.6),
        whiskerprops=dict(linewidth=1.4),
        capprops=dict(linewidth=1.4),
    )
    # 填充颜色（浅灰）
    for b in bp["boxes"]:
        b.set_facecolor((0.92, 0.92, 0.92))
        b.set_edgecolor("black")

    # jitter points（抽样，避免太密）
    rng = np.random.default_rng(0)
    def jitter_scatter(vals, x0):
        vals = np.asarray(vals)
        if vals.size > show_points:
            vals = rng.choice(vals, size=show_points, replace=False)
        xs = x0 + rng.normal(0, 0.055, size=vals.size)
        ax.scatter(xs, vals, s=point_size, alpha=point_alpha, linewidths=0)

    jitter_scatter(gNR, 1)
    jitter_scatter(gR,  2)

    # labels
    ax.set_xticks([1, 2])
    ax.set_xticklabels([f"Non-Responder\n(n={len(gNR)})", f"Responder\n(n={len(gR)})"], fontsize=11)
    ax.set_ylabel(f"regulon AUC: {tf_label}", fontsize=12)

    # 标题 + 统计注释
    ax.set_title(f"{title_prefix}: {tf_label}", fontsize=13, pad=10)
    subtitle = f"Mann–Whitney p={p:.2e}   Cliff's δ={d:.2f}   medianΔ={medNR-medR:.3g}"
    ax.text(0.02, 1.02, subtitle, transform=ax.transAxes, fontsize=10, va="bottom")

    # 美化坐标轴
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", labelsize=11)

    if ylim is not None:
        ax.set_ylim(*ylim)

    plt.tight_layout()
    if outpath:
        fig.savefig(outpath, dpi=250, bbox_inches="tight")
    plt.close(fig)

    return {"n_NR": len(gNR), "n_R": len(gR), "p": p, "delta": d, "status": "OK"}


In [ ]:
mask_c3 = ad.obs["leiden_T_0.6_relabel"].astype(str).isin(["3"])
group = ad.obs.loc[mask_c3, "Respond"].astype(str).values   # 这里是 array: Non-Responder/Responder
X_c3 = ad.obsm["X_regulon_auc"].loc[mask_c3]                 # C3 cells x regulons
y = X_c3["FOXO3(+)"].astype(float).to_numpy()
res = boxplot_pretty_two_groups(
    y=y,                        # y 也必须是 mask_c3 对应的长度
    group=group,
    tf_label="FOXO3(+)",
    outpath="/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/10_TF_analysis/figures/C3_boxplot_FOXO3.png"
)
print(res)

In [ ]:
print("mask_c3 sum =", int(mask_c3.sum()))
print("len(group)  =", len(group))
print("len(y)      =", len(y))

In [ ]:
import os
import numpy as np
import pandas as pd

# 你前面已经有这两个函数的话，就不用重复定义：
# - resolve_tf_to_cols(X_df, tf, prefer_sign="+")
# - boxplot_pretty_two_groups(y, group, tf_label, outpath, ...)

def plot_NR_gt_R_TFs_in_C3(
    ad,
    out_dir,
    tf_list,
    cluster_col="leiden_T_0.6_relabel",
    c3_labels=("3","C3"),
    group_col="Respond",              # 这里是 Non-Responder / Responder
    prefer_sign="+",
    title_prefix="C3 regulon activity (Non-Responder vs Responder)"
):
    os.makedirs(out_dir, exist_ok=True)

    mask_c3 = ad.obs[cluster_col].astype(str).isin(list(c3_labels))
    X_c3 = ad.obsm["X_regulon_auc"].loc[mask_c3]
    group = ad.obs.loc[mask_c3, group_col].astype(str).values

    summary = []
    for tf in tf_list:
        cols = resolve_tf_to_cols(X_c3, tf, prefer_sign=prefer_sign)
        if len(cols) == 0:
            summary.append({"TF": tf, "used_col": "", "status": "SKIP_no_match"})
            continue

        # 如果命中多个列，取均值（一般只会命中一个 TF(+)）
        y = X_c3[cols].astype(float).mean(axis=1).to_numpy()

        out_png = os.path.join(out_dir, f"C3_boxplot_{tf}_NRgtR.png")
        res = boxplot_pretty_two_groups(
            y=y,
            group=group,
            tf_label=cols[0],          # 用真实列名，比如 FOXO3(+)
            outpath=out_png,
            title_prefix=title_prefix
        )
        summary.append({
            "TF": tf,
            "used_col": ";".join(cols),
            "png": out_png,
            **res
        })

    summary_df = pd.DataFrame(summary)
    # 按 p 值排序（OK 的在前）
    if "p" in summary_df.columns:
        summary_df = summary_df.sort_values(["status","p"], ascending=[True, True])
    summary_path = os.path.join(out_dir, "C3_NRgtR_TF_boxplots_summary.csv")
    summary_df.to_csv(summary_path, index=False)
    print("[DONE] saved summary:", summary_path)
    return summary_df


# ------- 你要画的 TF 列表（NR > R 候选）-------
tf_list = ["FOXO3","BATF","AHR","RELB","NFKB1","NFATC1","NR1D2","ARNTL"]

out_dir = "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/10_TF_analysis/figures/C3_NRgtR_TFs/"
summary_df = plot_NR_gt_R_TFs_in_C3(ad, out_dir=out_dir, tf_list=tf_list)
summary_df.head(12)


In [ ]:
tf_list_RgtNR = ["EOMES","PRDM1","STAT1","NFATC2","IRF8","TBX21"]
out_dir = "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/10_TF_analysis/figures/C3_RgtNR_TFs/"
summary_df_R = plot_NR_gt_R_TFs_in_C3(ad, out_dir=out_dir, tf_list=tf_list_RgtNR)

In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

In [ ]:
PERT_H5AD = r"/ShangGaoAIProjects/Zang/single_cell_data/scRNA_reference_dataset/T_cell_perturb_seq/GWCD4i.DE_stats.h5ad"
